# Clinical scores, leakage audit and patient-level validation

This notebook evaluates the existing scores directly, audits feature availability, and validates one model with patient-grouped cross-validation.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import yaml
import pandas as pd
from icare_risk.data.audit import load_admission_file
from icare_risk.evaluation.raw_scores import evaluate_raw_scores
from icare_risk.explainability.feature_availability import audit_feature_availability
from icare_risk.evaluation.patient_validation import grouped_cross_validate_model

In [3]:
CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)
df = load_admission_file(config['input']['path'])
target = config['columns']['target']
score_columns = {n: s['source_column'] for n, s in config.get('scores', {}).items()}
score_results = evaluate_raw_scores(df, score_columns, target, target_mapping=config.get('target_mapping', {}))
display(score_results.sort_values('auroc', ascending=False))

,score,source_column,n,n_events,auroc,auprc
5,holmgren_score,holmgren_score,10000,2049,0.500000,0.204900
4,gavaghan_score,gavaghan_score,10000,2049,0.499930,0.204881
2,tumbarello_score,tumbarello_score,10000,2049,0.496296,0.203872
1,kim_score,kim_score,10000,2049,0.496291,0.203857
3,jones_score,jones_score,10000,2049,0.494991,0.203600
0,charlson_score,charlson_score,10000,2049,0.488685,0.200932


## Feature availability and leakage audit

In [4]:
feature_audit = audit_feature_availability(
    df, config.get('feature_sets', {}),
    outcome_columns=[target, 'esbl_status', 'esbl_label_test'],
    post_prediction_columns=['closest_culture_date', 'time_to_culture_days', 'time_to_culture_hours', 'qualifying_last_abx', 'DISCHARGE_DATE', 'DEATH_DATE', 'UPDATE_DT_TM'],
)
display(feature_audit)

,feature_set,feature,present,decision,reason,prediction_time
0,charlson_score,charlson_hx_mi,True,review,requires clinical confirmation,admission
1,charlson_score,charlson_hx_chf,True,review,requires clinical confirmation,admission
2,charlson_score,charlson_hx_pvd,True,review,requires clinical confirmation,admission
3,charlson_score,charlson_hx_stroke,True,review,requires clinical confirmation,admission
4,charlson_score,charlson_hx_dementia,True,review,requires clinical confirmation,admission
...,...,...,...,...,...,...
239,all_available_baseline,ETHNICITY_DES,True,review,requires clinical confirmation,admission
240,all_available_baseline,GENDER,True,review,requires clinical confirmation,admission
241,all_available_baseline,GENDER_DESC,True,review,requires clinical confirmation,admission
242,all_available_baseline,UPDATE_DT_TM,True,exclude,unavailable after prediction time,admission


## Patient-level validation

In [5]:
model_cfg = next(m for m in config['models'] if m['name'] == 'lr_clinical_baseline')
features = config['feature_sets'][model_cfg['feature_set']]['columns']
grouped_results = grouped_cross_validate_model(
    df[df['split'].isin(['train', 'validation'])], features, target,
    config['columns']['patient_id'], model_type=model_cfg['type'],
    params=model_cfg.get('params', {}), target_mapping=config.get('target_mapping', {}),
)
display(grouped_results)

KeyError: 'split'